<a href="https://colab.research.google.com/github/Sathvara-Amitkumar/RNN_Next-Word-Prediction/blob/main/next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
cd /content/drive/MyDrive/Colab Notebooks/RNN/

/content/drive/MyDrive/Colab Notebooks/RNN


In [4]:
ls

next_word_predictor.ipynb  qoute_dataset.csv


In [5]:
df = pd.read_csv('qoute_dataset.csv')

In [6]:
df.shape

(3038, 2)

In [7]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [8]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [9]:
quotes = df['quote']
quotes

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."
...,...
3033,The past beats inside me like a second heart.
3034,"Damn, Claire. Warn a guy before you do a face-..."
3035,"Can you be a girl for a few seconds?""""I'm alwa..."
3036,That's what fiction is for. It's for getting a...


In [10]:
quotes = quotes.str.lower()

In [11]:
import string
translater = str.maketrans(' ', ' ', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translater))

In [12]:
quotes.sample(5)

,quote
1126,shed laughed and if he could have bottled the ...
1673,sometimes when i look at you i feel im gazing ...
2598,the things you used to own now they own you
740,you could have had anything else in the world ...
2931,whats up i askedyou tell me he said you were t...


## Tokenizer

In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [14]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [15]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:15]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10),
 ('in', 11),
 ('be', 12),
 ('not', 13),
 ('are', 14),
 ('your', 15)]

### Convert Text to Sequences

Now, let's use the fitted tokenizer to transform our text quotes into sequences of token IDs.

In [16]:
sequences = tokenizer.texts_to_sequences(quotes)
print(f"Number of sequences: {len(sequences)}")
print("Sample original quote:", quotes.iloc[0])
print("Sample tokenized sequence:", sequences[0])

Number of sequences: 3038
Sample original quote: “the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
Sample tokenized sequence: [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]


In [25]:
for i in range(5):
  print(quotes.iloc[i])
  print(sequences[i], "\n")

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461] 

“it is our choices harry that show what we truly are far more than our abilities”
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676] 

“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677] 

“the person be it gentleman or lady who has not pleasure in a good novel must be intolerably stupid”
[713, 116, 12, 10, 2462, 32, 1043, 30, 82, 13, 601, 11, 5, 74, 1338, 119, 12, 2463, 3678] 

“imperfection is beauty madness is genius and its better to be absolutely ridiculous than absolutely boring”
[3679, 7, 313, 753, 7, 638, 4, 43, 144, 3, 12, 682, 1339, 54, 682, 3680] 



### Create N-Gram Input and Target Sequences

We will iterate through our tokenized `sequences` to build input-target pairs ($X$ and $y$) for our next-word prediction model.

In [49]:
input_sequences = []

# Loop over each sequence of token IDs to create incremental n-grams
for seq in sequences:
    for i in range(1, len(seq)):
        n_gram_sequence = seq[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total generated n-gram sequences: {len(input_sequences)}")

# To make it ready for training, we'll pad the sequences to have uniform length
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Splitting into predictors (X) and label (y)
X = input_sequences[:, :-1]
Y = input_sequences[:, -1]

print(f"X shape: {X.shape}")
print(f"y shape: {Y.shape}")

Total generated n-gram sequences: 85271
X shape: (85271, 745)
y shape: (85271,)


In [50]:
X

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

### One-Hot Encoding the Target Variable

To train our multi-class classification model, we need to convert our target class labels ($Y$) into a one-hot encoded representation. Since our word indices start at `1` and can go up to `len(word_index) + 1` (to account for the out-of-vocabulary padding or index 0), we will define the total number of classes accordingly.

In [52]:
from tensorflow.keras.utils import to_categorical

# Total number of unique classes
num_classes = len(word_index) + 1

# Perform one-hot encoding
Y_encoded = to_categorical(Y, num_classes=num_classes)

print(f"Original Y shape: {Y.shape}")
print(f"One-hot encoded Y shape: {Y_encoded.shape}")

Original Y shape: (85271,)
One-hot encoded Y shape: (85271, 8979)


In [56]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

### Build and Compile SimpleRNN Model

We will define a simple sequential model starting with an `Embedding` layer, followed by a `SimpleRNN` layer, and finally a output `Dense` layer with a softmax activation function to predict the next word.

In [57]:
# Define model parameters
embedding_dim = 50
input_length = X.shape[1]  # Which is max_sequence_len - 1 (745)

# Build the SimpleRNN model
rnn_model = Sequential([
    Embedding(input_dim=num_classes, output_dim=embedding_dim, input_length=input_length),
    SimpleRNN(units=128, return_sequences=False),
    Dense(units=num_classes, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [58]:
# Compile the model
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [59]:
# Print model summary
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [61]:
# Define LSTM model parameters
embedding_dim = 50
input_length = X.shape[1]

# Build the LSTM model
lstm_model = Sequential([
    Embedding(input_dim=num_classes, output_dim=embedding_dim, input_length=input_length),
    LSTM(units=128, return_sequences=False),
    Dense(units=num_classes, activation='softmax')
])

In [62]:
# Compile the LSTM model
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)